# 04 — Inventory Optimization
**Module D**

Monte Carlo simulation for optimal reorder policy:
- EOQ baseline computation
- 10,000 simulations per (R, Q) combination
- Grid search across safety stock levels and order quantities
- Three scenarios: baseline, moderate disruption, severe disruption
- Cost breakdown: holding, ordering, stockout

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from src.optimize import run_module_d
from src.viz import create_monte_carlo_distribution

In [2]:
results = run_module_d()

╔══════════════════════════════════════════════════════╗
║  Module D — Inventory Optimization                   ║
╚══════════════════════════════════════════════════════╝

  Optimizing 50 representative SKUs across 3 scenarios...


    ... optimized 10/50 SKUs


    ... optimized 20/50 SKUs


    ... optimized 30/50 SKUs


    ... optimized 40/50 SKUs


    ... optimized 50/50 SKUs

  ── Optimization Summary (Baseline Scenario) ──
     Average EOQ: 110 units
     Average optimal reorder point: 647 units
     Average safety stock: 6.8 weeks
     Average annual cost: $13,850,775
     Average stockout weeks: 10.4

  ── Disruption Risk Tiers ──
     High: 17 SKUs (avg cost increase: 72607.8%)
     Medium: 6 SKUs (avg cost increase: 30753.0%)
     Low: 27 SKUs (avg cost increase: 0.0%)

  ✓ Optimization results saved to /Users/harthikmallichetty/Desktop/global-supply-chain-intelligence/notebooks/../data/processed/optimization_results.pkl


## Optimization Results

In [3]:
# Baseline scenario results
baseline = results['results'][results['results']['scenario'] == 'baseline']
baseline[['sku_id', 'category', 'eoq', 'optimal_reorder_point', 
          'optimal_order_quantity', 'safety_stock_weeks', 
          'expected_annual_cost']].head(20)

,sku_id,category,eoq,optimal_reorder_point,optimal_order_quantity,safety_stock_weeks,expected_annual_cost
0,SKU-0362,Metals,55,426,165,7.99,12613267.80
3,SKU-0074,Electronics,173,545,519,4.44,11877.55
6,SKU-0375,Metals,58,464,174,7.98,10902173.33
9,SKU-0156,Auto Parts,61,308,183,3.55,31943.07
12,SKU-0105,Auto Parts,154,388,308,4.43,6694.34
15,SKU-0395,Pharmaceuticals,48,649,144,7.11,118854.91
18,SKU-0378,Metals,80,181,120,3.55,7334.70
21,SKU-0125,Auto Parts,140,611,420,7.99,4928236.40
24,SKU-0069,Electronics,66,1114,198,8.00,65098238.33
27,SKU-0451,Textiles,282,497,564,6.22,2914.00


## Scenario Comparison

In [4]:
import plotly.express as px

scenario_summary = results['results'].groupby('scenario').agg(
    avg_cost=('expected_annual_cost', 'mean'),
    avg_safety_stock=('safety_stock_weeks', 'mean'),
    avg_stockout_weeks=('mean_stockout_weeks', 'mean'),
).reset_index()

fig = px.bar(scenario_summary, x='scenario', y='avg_cost', 
             color='scenario', title='Average Annual Cost by Scenario',
             color_discrete_map={'baseline': '#10B981', 'moderate': '#F59E0B', 'severe': '#EF4444'})
fig.update_layout(plot_bgcolor='#0D0D0D', paper_bgcolor='#0D0D0D', font=dict(color='white'))
fig.show()

scenario_summary

,scenario,avg_cost,avg_safety_stock,avg_stockout_weeks
0,baseline,1.385078e+07,6.8384,10.4456
1,moderate,1.809121e+07,7.8346,20.4672
2,severe,1.910545e+07,7.9944,27.8580


## Disruption Risk Tiers

In [5]:
risk = results['cost_comparison'].sort_values('cost_increase_pct', ascending=False)
risk.head(15)

,sku_id,baseline_cost,severe_cost,cost_increase_pct,disruption_risk_tier
46,SKU-0001,9342.82,14117089.33,151000.95,High
28,SKU-0102,13121.59,15460737.07,117726.70,High
6,SKU-0378,7334.70,8547354.20,116433.11,High
26,SKU-0498,5890.51,5756755.20,97629.32,High
1,SKU-0074,11877.55,9622424.80,80913.55,High
4,SKU-0105,6694.34,5070148.53,75637.84,High
14,SKU-0372,11635.79,8763029.33,75211.00,High
35,SKU-0334,18421.01,11992896.00,65004.44,High
22,SKU-0281,6506.40,4096001.60,62853.42,High
3,SKU-0156,31943.07,18552144.00,57978.78,High
